In [1]:
import glob

import numpy as np
import pandas as pd

from shutil import copy2

In [2]:
coords = pd.read_csv('../inputs/county_locations.csv', dtype={'fips': str, 'x': float, 'y': float})

In [3]:
files = glob.glob('*.csv~')

In [4]:
for file in files:
    # make backup
#     copy2(file, '%s~' % file)
    
    # load
    df = pd.read_csv(file, dtype={'region_production': str, 'region_destination': str})
    
    # left-pad FIPS
    df.region_production = df.region_production.str.pad(width=5, side='left', fillchar='0')
    df.region_destination = df.region_destination.str.pad(width=5, side='left', fillchar='0')

    # replace outdated FIPS
    df.loc[df.region_production == '51515', 'region_destination'] = '13115'  # 51515 was decomissioned as an independent city and absorbed into 51019 in 2012; setting destination to existing destination for 51019
    df.loc[df.region_production == '51515', 'region_production'] = '51019'  # was decomissioned as an independent city and absorbed into 51019 in 2012
    df.loc[df.region_production == '46113', 'region_production'] = '46102'  # renamed and renumbered in 2015
    
    # backfill NULLs
    df.fillna('-1', inplace=True)
    
    # dissolve 
    df = df.groupby(['feedstock', 'tillage_type', 'region_production', 'region_destination', 'equipment_group', 'feedstock_measure', 'unit_numerator', 'unit_denominator']).sum().reset_index()[['feedstock', 'tillage_type', 'region_production', 'region_destination', 'equipment_group', 'feedstock_measure', 'unit_numerator', 'feedstock_amount', 'unit_denominator']]

    # reinstate NULLs
    df.replace('-1', np.nan, inplace=True)
    
    # get coordinates for production
    df = df.merge(coords, how='left', left_on='region_production', right_on='fips')
    
    # drop extraneous columns
    df.drop(['fips'], axis=1, inplace=True)

    # rename source columns
    df.rename(columns={'x': 'source_lon', 'y': 'source_lat'}, inplace=True)
    
    # get coordinates for destination
    df = df.merge(coords, how='left', left_on='region_destination', right_on ='fips')
    
    # drop extraneous columns
    df.drop(['fips'], axis=1, inplace=True)

    # rename destination columns
    df.rename(columns={'x': 'destination_lon', 'y': 'destination_lat'}, inplace=True)

    # save to CSV
    df.to_csv(file.replace('~', ''), header=True, index=False, float_format='%.5f')


In [5]:
files = glob.glob('*.csv')

In [6]:
for file in files:
    print(file)
    df = pd.read_csv(file)
    print(df[['source_lon', 'source_lat', 'destination_lon', 'destination_lat']].isna().sum())

production_2015_bc1060.csv
source_lon         0
source_lat         0
destination_lon    0
destination_lat    0
dtype: int64
production_2015_hh3060.csv
source_lon         0
source_lat         0
destination_lon    0
destination_lat    0
dtype: int64
production_2017_bc1060.csv
source_lon         0
source_lat         0
destination_lon    0
destination_lat    0
dtype: int64
production_2017_hh3060.csv
source_lon         0
source_lat         0
destination_lon    0
destination_lat    0
dtype: int64
production_2017_hhhe.csv
source_lon         0
source_lat         0
destination_lon    0
destination_lat    0
dtype: int64
production_2017_mhle.csv
source_lon         0
source_lat         0
destination_lon    0
destination_lat    0
dtype: int64
production_2040_bc1060.csv
source_lon         0
source_lat         0
destination_lon    0
destination_lat    0
dtype: int64
production_2040_hh3060.csv
source_lon         0
source_lat         0
destination_lon    0
destination_lat    0
dtype: int64
production_2

In [50]:
files = glob.glob('*.csv')

In [55]:
for file in files:
    df = pd.read_csv(file, dtype={'region_production': str, 'region_destination': str})
    
    print(df.loc[df.region_production == '51019']['feedstock_amount'].describe())

count        64.000000
mean       4312.372583
std       16438.945653
min           1.194730
25%          62.963685
50%         231.409890
75%         812.886462
max      110379.376400
Name: feedstock_amount, dtype: float64
count        72.000000
mean       3870.134555
std       15536.771592
min           0.839000
25%          53.117765
50%         224.465685
75%         802.604100
max      110379.376400
Name: feedstock_amount, dtype: float64
count        64.000000
mean       4285.231213
std       16333.865008
min           1.220280
25%          59.294220
50%         225.979655
75%         772.750123
max      109912.105300
Name: feedstock_amount, dtype: float64
count        72.000000
mean       3845.580413
std       15437.937933
min           0.886000
25%          51.292795
50%         216.080360
75%         745.358000
max      109912.105300
Name: feedstock_amount, dtype: float64
count        4.000000
mean     22458.675000
std      31555.314166
min       2594.400000
25%       2986.35000